# project_15_affinity_maturation — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — affinity maturation + developability theory, CDR contacts, the mock hello-world

**Standard slot:** *define & explore.* **For Project 15 this means:** you are NOT designing an
antibody de novo — you are **lead-optimizing an EXISTING antibody**. Start from a *known*
antibody-antigen complex (with a **measured KD in the literature**), understand which CDR residues
contact the antigen, fix the metrics table, and run the **mock** affinity-maturation hello-world
end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend — switch to the real backends (ESM-1v / AbLang / ProteinMPNN / AF2-Multimer) on Colab
in notebooks 02–04.

## The problem in one screen

**Affinity maturation, computationally.** A therapeutic-antibody lead usually *binds*, but not tightly
or cleanly enough. **Lead optimization** — raising affinity while keeping **developability** (no
aggregation, no chemical-liability hotspots, expressible, stable) — is slow and expensive in the lab.
The computational job is to propose a **small, testable set** of CDR mutations that are *likely* to
improve the antibody, so the wet lab tests ~10 variants instead of thousands.

**Why "existing antibody", not de novo.** You inherit a real paratope and a real, measured starting
affinity. Mutations are **edits** to that paratope:
- **Framework is FIXED.** Only **CDR** positions (the loops that contact antigen) are varied — that is
  what "maturation" means. Touching the framework risks folding/expression and humanness.
- **CDR3** dominates the paratope (longest, most diverse loop); CDR1/CDR2 contribute too.

**The two honesty rules for this whole project:**
1. **Never fabricate KD / ΔΔG / affinity numbers.** The deliverable is a **ranked, ordered** set of
   candidate mutations + the **experiment** (SPR/DSF) that would test them — not predicted constants.
   Every mock number here is a dimensionless **ranking score**, flagged `SYNTHETIC`.
2. **Most predicted affinity-improving mutations do NOT validate.** Output a SMALL ranked set with
   mandatory **controls** (WT baseline + a destabilizing decoy). Report the rate, not the cherry.

## The metrics table (what we will score and filter on)

| Metric | Range | Means | Does **not** mean | Used for |
|--------|-------|-------|-------------------|----------|
| ESM-1v Δ-log-likelihood | ~[-3,+3] | protein-LM favors the mutation over WT (> 0) | higher affinity / a ΔΔG | RANK single mutations |
| AbLang naturalness | 0–1 | antibody-specific "natural-looking" prior | low immunogenicity / affinity | sanity-check CDR sequences |
| pae_interaction | Å | AF2-Multimer confidence in the **interface** pose | binding/affinity | **pose maintenance** (≤ 12) |
| scRMSD (vs parent) | Å | variant Fv backbone vs the parent pose | binding | **pose maintenance** (≤ 3.0) |
| developability liabilities | count | CDR chemical-liability motifs (NG/DG, Met-ox, free Cys, sequon) | a real TAP verdict | triage **before** synthesis |

The `"antibody"` cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12) come from the shared
`filtering_pipeline.DEFAULT_CUTOFFS["antibody"]`. **ESM-1v/AbLang scores RANK candidates; they are NOT
affinities. Developability here is a TEACHING HEURISTIC, not the validated tool (TAP/CamSol/real
deamidation predictors)** — see `MANUAL.md §2` and `maturation_tools.py`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your antibody-antigen complex (with a published KD)

The single most important input is a **real antibody-antigen complex from SAbDab** that has a
**measured KD reported in the literature** — that KD is your starting affinity, the baseline every
proposed mutation is measured against. The accession below is a **candidate placeholder — verify it on
SAbDab/RCSB in Week 1, and confirm a published KD exists for it.** Do not assert any KD value here.

In [ ]:
# --- Campaign definition (EDIT in Week 1 after choosing + verifying your complex) ---
# Pick a well-characterized therapeutic Fab-antigen complex from SAbDab WITH a published KD.
COMPLEX_PDB = "VERIFY_ON_SABDAB"   # e.g. a therapeutic Fab-antigen complex (candidate — verify; needs a published KD)
ANTIGEN = "ANTIGEN"                 # name your antigen once the complex is chosen
PARENT_KD_NOTE = ("the parent KD is the MEASURED literature value for your chosen complex — record the "
                  "exact value + its citation in your problem statement. Do NOT invent a number here.")

print("complex (candidate — verify on SAbDab/RCSB):", COMPLEX_PDB)
print("antigen :", ANTIGEN)
print("parent KD:", PARENT_KD_NOTE)

## The parent antibody + its CDRs

`maturation_tools.EXAMPLE_FRAMEWORK` + `example_parent_sequence()` are a **teaching placeholder** so the
plumbing runs anywhere. In Week 1 you **replace** them with the chains read off your verified complex,
and you derive the CDR boundaries with a real antibody numbering scheme (IMGT/Kabat/Chothia via ANARCI)
— do not eyeball them. The framework is **fixed**; the CDR spans below are the only positions you may
mutate.

In [ ]:
from maturation_tools import (EXAMPLE_FRAMEWORK, example_parent_sequence, cdr_positions)

parent = example_parent_sequence()      # EXAMPLE placeholder VH — replace with your real chain
spans = cdr_positions()
print("framework:", EXAMPLE_FRAMEWORK["name"], "(teaching placeholder — replace with your real framework)")
print("parent length:", len(parent), "aa")
print("CDR spans (0-based, half-open) — the ONLY mutable positions:")
for cdr, (a, b) in spans.items():
    print(f"  {cdr}: residues {a+1}-{b}  loop = {parent[a:b]}")

## Identify CDR contact residues (the maturation targets)

Affinity maturation focuses mutations on residues that **contact the antigen** (or support contacting
loops). On a real complex you compute the **paratope** = antibody residues within ~4–5 Å of any antigen
atom (Biopython `NeighborSearch` on your verified PDB). Here, with no structure loaded, we mark the CDR
spans as the candidate region and leave the real contact extraction as a TODO for notebook 02 / Week 4.
Targeting *contact* residues (not all CDR residues) keeps the candidate set small and physically
motivated.

In [ ]:
# On a real complex, replace this with a contact calculation from the PDB:
#   from Bio.PDB import PDBParser, NeighborSearch
#   parse COMPLEX_PDB -> antibody atoms + antigen atoms
#   paratope = {antibody residues with any atom within 4.5 A of any antigen atom}
#   intersect paratope with the CDR spans -> the CONTACT residues you prioritise.
# For the mock hello-world we treat the CDR spans as the candidate region.
contact_region = []
for cdr, (a, b) in spans.items():
    contact_region += list(range(a + 1, b + 1))   # 1-based positions
print("candidate (CDR) positions to consider for mutation:", contact_region)
print("TODO (Week 4): intersect with the real paratope (<=4.5 A contacts) from your verified complex.")

## Affinity-maturation hello-world (mock backend, no GPU)

Score a few single CDR mutations with the ESM-1v / AbLang proxies, assemble a tiny ranked candidate
set, then pose-check + developability-scan it. This proves the plumbing (score → rank → pose check →
liability scan → controls) before any GPU time in notebooks 02–04. **Every number below is a SYNTHETIC
ranking score — never a KD, never report it as a real affinity.**

In [ ]:
from maturation_tools import (score_single_mutations, assemble_candidate_set,
                              score_variants, make_controls)

# Score every single substitution at every CDR position (framework fixed), then rank a SMALL set.
singles = score_single_mutations(parent, tool="mock")
cand = assemble_candidate_set(singles, top_n=5)
score_variants(cand, tool="mock")        # fills af2 pose check + developability (SYNTHETIC)

print(f"scored {len(singles)} single CDR mutations; showing the top-5 RANKED candidates (NOT KDs):\n")
print(f"{'mutation':10s} {'CDR':5s} {'esm1v':>7s} {'ablang':>7s} {'pae':>5s} {'scrmsd':>7s} {'liab':>5s}")
for v in cand:
    print(f"{'+'.join(v.mutations):10s} {v.cdr:5s} {v.esm1v:>+7.3f} {v.ablang:>7.3f} "
          f"{v.pae_interaction:>5.1f} {v.scrmsd:>7.3f} {v.n_liabilities:>5d}")

ctrls = make_controls(parent, antigen=ANTIGEN)
print("\ncontrols (mandatory):", [c.design_id for c in ctrls])
print("synthetic:", cand[0].synthetic, "->", cand[0].notes[0] if cand[0].notes else "")

## Visualize the complex (py3Dmol)

Use this to eyeball your real antibody-antigen complex and its paratope once you have the verified PDB.
The mock backend writes no structure.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after fetching your verified complex with data/download_data.py):
# show_pdb("data/inputs/<YOUR_COMPLEX>.pdb")
print("show_pdb(pdb_path) ready — use it on your verified antibody-antigen complex.")

## D0 checklist
- [ ] 1-page **problem statement**: the chosen **SAbDab complex** (verified accession), its **measured
      literature KD** (value + citation — not invented), the CDR contact residues you will target, and
      **measurable** success criteria.
- [ ] Verified the complex on SAbDab/RCSB; confirmed a **published KD** exists; derived CDR boundaries
      with a real numbering scheme (not the EXAMPLE placeholder).
- [ ] Metric table understood, including that ESM-1v/AbLang are **ranking** signals (not affinity) and
      developability here is a **heuristic**.
- [ ] Mock hello-world run; top-ranked single mutations + SYNTHETIC pose/liability metrics printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — the single-mutation scoring + ProteinMPNN CDR-redesign campaign.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — single-mutation scoring + ProteinMPNN CDR redesign → candidate CSV

**Standard slot:** *design campaign.* **For Project 15 this means:** the maturation campaign —
(a) **score single CDR mutations** with ESM-1v and AbLang (framework fixed), (b) run **ProteinMPNN CDR
redesigns** (framework fixed) to catch multi-residue loop changes, and (c) assemble a **candidate
mutation set** as a results CSV (D2).

**Compute reality (be honest):** ESM-1v and AbLang scoring is **light** — CPU/T4 is fine, and this is
the cheap, high-value part. ProteinMPNN CDR redesign is also cheap. The **heavy** step (notebook 03/04)
is AF2-Multimer pose checking — keep N small and **batch** it. Colab **T4–Pro** is the realistic tier.
This notebook runs on the **mock** backend so the plumbing executes anywhere; all numbers are
RANKING scores, flagged SYNTHETIC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything.

In [ ]:
import requests

# Pinned upstreams for affinity maturation (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "ESM / ESM-1v (protein LM mutation scoring)": "https://github.com/facebookresearch/esm",
    "AbLang (antibody LM)": "https://github.com/oxpig/AbLang",
    "ProteinMPNN (CDR redesign, framework fixed)": "https://github.com/dauparas/ProteinMPNN",
    "ColabFold (AF2-Multimer pose check)": "https://github.com/sokrypton/ColabFold",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: AbLang2 supersedes AbLang — VERIFY the current public repo and pin it (MANUAL.md §2).")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters

Re-state the parent + the campaign scale. The single-mutation scan is exhaustive over the CDR positions
(cheap). ProteinMPNN redesigns add multi-residue loop variants. Keep the eventual *tested* set SMALL —
diversity in scoring, then aggressive filtering, then a short list to the bench.

In [ ]:
from maturation_tools import (example_parent_sequence, cdr_positions, EXAMPLE_FRAMEWORK)

ANTIGEN = "ANTIGEN"
parent = example_parent_sequence()       # EXAMPLE placeholder — use your verified chain on a real run
FRAMEWORK = EXAMPLE_FRAMEWORK

# Single-mutation scan is exhaustive over CDR positions; MPNN adds N redesigns per CDR.
MPNN_PER_CDR = 8         # small mock N; real ProteinMPNN run can sample more (cheap)
TOOL = "mock"            # -> "esm1v"/"ablang"/"proteinmpnn" on Colab (light; CPU/T4 fine)

print("parent length:", len(parent), "aa")
print("CDR spans:", cdr_positions(FRAMEWORK))
print(f"campaign: exhaustive single-mutation scan + {MPNN_PER_CDR} MPNN redesigns/CDR  (tool={TOOL})")
print("Diversity BEFORE filtering: score broadly here, filter hard in nb 03, test a SMALL set.")

## (a) Single-mutation scoring (ESM-1v + AbLang)

`score_single_mutations()` tries the 19 non-WT residues at every CDR position (framework fixed) and
scores each with the ESM-1v Δ-log-likelihood proxy + AbLang naturalness. Switch `TOOL` to `"esm1v"` /
`"ablang"` on Colab to run for real (the functions raise a clear, actionable `NotImplementedError` with
the TODO until then). **These are RANKING scores, not affinities.**

In [ ]:
import pandas as pd
from maturation_tools import score_single_mutations

singles = score_single_mutations(parent, framework=FRAMEWORK, tool=TOOL, antigen=ANTIGEN)
rows = [v.as_row() for v in singles]
single_df = pd.DataFrame(rows)
print(f"scored {len(single_df)} single CDR mutations (ESM-1v + AbLang, {('SYNTHETIC' if TOOL=='mock' else 'real')})")
# Favored single mutations (esm1v > 0) are the maturation candidates; show the strongest few.
fav = single_df.sort_values("esm1v", ascending=False).head(8)
print("\ntop single mutations by ESM-1v (RANK only, NOT KD):")
fav[["design_id", "cdr", "mutations", "esm1v", "ablang"]]

## (b) ProteinMPNN CDR redesigns (framework FIXED)

`mpnn_cdr_redesign()` proposes new sequences for ONE CDR loop at a time while holding **all** framework
positions (and the other CDRs) fixed — exploring multi-residue changes that single-mutation scanning
misses. On a real run this uses a ProteinMPNN design mask so only the chosen CDR varies. Survivors must
still pass the pose check + developability scan downstream.

In [ ]:
from maturation_tools import mpnn_cdr_redesign

redesigns = []
for cdr in ("CDR1", "CDR2", "CDR3"):
    redesigns += mpnn_cdr_redesign(parent, cdr=cdr, n=MPNN_PER_CDR, framework=FRAMEWORK,
                                   tool=TOOL, antigen=ANTIGEN)
print(f"ProteinMPNN redesigns (framework FIXED): {len(redesigns)} "
      f"({MPNN_PER_CDR} per CDR x 3 CDRs)")
redesign_df = pd.DataFrame([v.as_row() for v in redesigns])
redesign_df[["design_id", "cdr", "mutations", "ablang"]].head()

## (c) Assemble the candidate mutation set → CSV

Combine the single mutations and the CDR redesigns into one candidate pool and write the campaign CSV.
We keep the columns the filter + analysis need. This pool is deliberately broad at SCORING time; the
aggressive filtering in notebook 03 and the SMALL final list happen later — diversity before
filtering.

In [ ]:
camp = pd.concat([single_df, redesign_df], ignore_index=True)
cols = ["design_id", "source", "antigen", "cdr", "mutations",
        "esm1v", "ablang", "pae_interaction", "scrmsd", "n_liabilities", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?", bool(camp["synthetic"].fillna(False).all()) if "synthetic" in camp else "n/a",
      "(mock => all scores are EXAMPLE_DATA ranking values, not affinities)")
camp.head()

## Quick campaign sanity look

Before filtering, eyeball the distributions: the ESM-1v score (how many mutations the LM favors at all)
and the AbLang naturalness. On the **mock** backend these are SYNTHETIC and only show the plumbing; on a
real run they tell you whether there is signal worth carrying into the pose check.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(camp["esm1v"].dropna(), bins=20); ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].set_title("ESM-1v Δ-LL (mock)"); ax[0].set_xlabel(">0 = favored over WT")
ax[1].hist(camp["ablang"].dropna(), bins=20); ax[1].set_title("AbLang naturalness (mock)")
ax[1].set_xlabel("0-1")
plt.suptitle("Campaign scores — SYNTHETIC (mock); for plumbing only; RANKS, not affinities")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock distributions are SYNTHETIC ranking scores — real shape comes from ESM-1v/AbLang.")

## D2 checklist
- [ ] `results/campaign.csv`: the candidate pool (single mutations + CDR redesigns), one row per
      candidate, with ESM-1v / AbLang ranking scores.
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md` (ESM, AbLang/AbLang2, MPNN).
- [ ] Design log: parent, framework, CDR spans, scan settings, MPNN params, seed, tool/version.
- [ ] (Real run) ESM-1v ensemble + AbLang scored; ProteinMPNN CDR redesigns with framework masked.
- [ ] Reminder recorded: these are **ranking** scores; the SMALL tested set + controls come later.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter (`design_type="antibody"`).

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 15** you map your candidate variants onto `fp.Design` objects, run the pipeline with
the **antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival
(D3 pt 1). The filter here enforces **pose maintenance** — a mutation that breaks the interface pose is
out, no matter how good its ESM-1v rank.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs are the same as the rest of the antibody family (Projects 14–17).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Score the candidates' pose + developability, then build `fp.Design` objects

The campaign CSV has ESM-1v/AbLang **ranking** scores but not yet the **pose-maintenance** metrics
(those need the heavier AF2-Multimer step). We fill `pae_interaction` + `scrmsd` (and the developability
liability count) with `score_variants()` — on `tool="mock"` these are SYNTHETIC; on Colab switch to
`tool="af2"`. Then we map each variant onto an `fp.Design`: for an antibody complex the key fields are
`plddt`, `pae_interaction`, and `scrmsd`. We stash ESM-1v/AbLang/liabilities in `extra` so they ride
along into the ranked CSV.

In [ ]:
from maturation_tools import (Variant, score_variants, example_parent_sequence)

camp = pd.read_csv("results/campaign.csv")

# Rebuild lightweight Variant objects from the CSV to run the (heavier) pose check + dev scan.
# (On a real run you would already have AF2-Multimer outputs; here we compute the mock metrics.)
parent = example_parent_sequence()
variants = []
for _, r in camp.iterrows():
    muts = tuple(str(r["mutations"]).split("+")) if isinstance(r.get("mutations"), str) and r["mutations"] else ()
    # reconstruct the mutated sequence for pose/dev scoring (single + simple combos)
    seq = parent
    ok = True
    try:
        from maturation_tools import apply_mutation
        for m in muts:
            seq = apply_mutation(seq, m)
    except Exception:
        ok = False
    v = Variant(design_id=str(r["design_id"]), sequence=seq, antigen=str(r.get("antigen", "ANTIGEN")),
                mutations=muts, cdr=str(r.get("cdr", "")), source=str(r.get("source", "mock")),
                esm1v=r.get("esm1v"), ablang=r.get("ablang"))
    if ok:
        variants.append(v)
score_variants(variants, tool="mock")    # fills pae_interaction, scrmsd, n_liabilities (SYNTHETIC)
print(f"scored pose + developability for {len(variants)} variants (SYNTHETIC on mock)")

In [ ]:
designs = []
for v in variants:
    designs.append(fp.Design(
        design_id=v.design_id,
        sequence="",                      # full chain not needed for the confidence layers
        design_type="antibody",
        plddt=70.0 if v.pae_interaction is not None else None,  # mock proxy; real run -> AF2 interface pLDDT
        pae_interaction=v.pae_interaction,
        scrmsd=v.scrmsd,
        # solubility maps to a developability proxy so Layer 3 (physics) has something to act on;
        # fewer liabilities => "more soluble/developable" in this teaching mapping:
        solubility=(-float(v.n_liabilities) if v.n_liabilities is not None else None),
        extra={"esm1v": v.esm1v, "ablang": v.ablang, "n_liabilities": v.n_liabilities,
               "mutations": "+".join(v.mutations), "cdr": v.cdr,
               "synthetic": bool(v.synthetic)},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")
print("NOTE: plddt here is a mock placeholder; on a real run use the AF2-Multimer interface pLDDT.")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency/pose + physics/developability); Layer 2
(orthogonal predictor agreement) needs a second predictor — wire an ESMFold/IgFold scRMSD in for the
real run. **On mock data the survivors are SYNTHETIC** — the point is the plumbing and the honest
accounting.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
top = fp.report(df_ranked, top_n=15, save_prefix="results/proj15")
print("\nranked CSV -> results/proj15_ranked.csv ; survival figure -> results/proj15_survival.png")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many candidates keep the binding **pose** (and stay developable) after filtering. For
affinity maturation this is the honest, expected funnel: many scored mutations, far fewer that maintain
the pose, and a **small** final set to test. Crucially, passing the filter means "still binds in the
same pose and is developable" — **NOT** "binds tighter". Only SPR can confirm tighter binding.

In [ ]:
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Candidates scored: {n_total}")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())
print("\nReminder: mock survivors are SYNTHETIC. Passing the filter = pose maintained + developable, "
      "NOT higher affinity. Only SPR (notebook 05) can confirm a real affinity gain.")

## D3 (part 1) checklist
- [ ] `results/proj15_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved); pose-maintenance enforced.
- [ ] Mapping assumptions written down (which metric → which `fp.Design` field; the plddt caveat).
- [ ] Honest hit-rate accounting; survivors framed as "pose-maintained + developable", not "tighter".

**Next:** `04_validate.ipynb` — pose maintenance, developability liability scan, epitope/epistasis.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — pose maintenance, developability liability scan, epistasis

**Standard slot:** *validate (in silico).* **For Project 15 the core analyses are:** (1) **pose
maintenance** — which candidates keep the parent binding pose (AF2-Multimer `pae_interaction` + scRMSD
vs parent), (2) the **developability liability scan** (flag NG/DG deamidation, Met oxidation, unpaired
Cys, glyc sequons introduced into the CDRs), and (3) **epistasis / combination** reasoning — do
single-mutation gains add up, or interfere? (D3 pt 2).

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC ranking values** — the
figures demonstrate the analysis; real numbers come from ESM-1v/AbLang + a batched AF2-Multimer run.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Pose maintenance — does the mutation keep the binding pose? `[core]`

The decisive maturation filter is not "is the ESM-1v score high" but "does the antibody still dock the
antigen the same way". Plot each candidate's `pae_interaction` (interface confidence; want ≤ 12) against
its `scrmsd` vs the parent pose (want ≤ 3.0). The good quadrant is **low-pae, low-scrmsd**. A high
ESM-1v rank with a broken pose is a false lead — this is exactly the cross-check a single-sequence
language model cannot do on its own.

In [ ]:
import pandas as pd
from maturation_tools import (example_parent_sequence, apply_mutation, Variant, score_variants)

camp = pd.read_csv("results/campaign.csv")
parent = example_parent_sequence()

# Rebuild + pose-check the top single-mutation candidates (by ESM-1v) for the figure.
top_single = camp[camp["source"] == "esm1v"].sort_values("esm1v", ascending=False).head(20)
vs = []
for _, r in top_single.iterrows():
    seq = parent
    try:
        for m in str(r["mutations"]).split("+"):
            if m:
                seq = apply_mutation(seq, m)
    except Exception:
        continue
    vs.append(Variant(design_id=str(r["design_id"]), sequence=seq, mutations=tuple(str(r["mutations"]).split("+")),
                      cdr=str(r["cdr"]), esm1v=r.get("esm1v"), ablang=r.get("ablang")))
score_variants(vs, tool="mock")
pose_df = pd.DataFrame([dict(design_id=v.design_id, mutation="+".join(v.mutations), cdr=v.cdr,
                            esm1v=v.esm1v, pae_interaction=v.pae_interaction, scrmsd=v.scrmsd,
                            n_liabilities=v.n_liabilities) for v in vs])
print("pose-maintenance table (SYNTHETIC mock metrics):")
pose_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4))
sc = ax.scatter(pose_df["scrmsd"], pose_df["pae_interaction"], c=pose_df["esm1v"], cmap="viridis", s=40)
ax.axvline(3.0, color="r", ls="--", lw=1, label="scrmsd cutoff 3.0")
ax.axhline(12, color="orange", ls="--", lw=1, label="pae cutoff 12")
ax.set_xlabel("scRMSD vs parent pose (Å)"); ax.set_ylabel("pae_interaction (Å)")
ax.set_title("Pose maintenance — SYNTHETIC (want low-left quadrant)")
plt.colorbar(sc, label="ESM-1v Δ-LL"); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("results/pose_maintenance.png", dpi=150); plt.show()
print("Good candidates: maintain the pose (low pae + low scrmsd) AND rank well on ESM-1v.")

## 2 · Developability liability scan `[core]`

A maturation mutation can quietly introduce a chemical liability into a CDR: an **NG/NS** Asn
**deamidation** hotspot, a **DG/DS** Asp **isomerization** hotspot, a solvent-exposed **Met** (oxidation)
or **Trp**, an **N-glycosylation sequon**, or an **unpaired Cys** (disulfide scrambling). These age the
drug, hurt manufacturability, and can sit right in the paratope. `developability_scan()` flags these
motifs **inside the CDRs**. It is a **TEACHING HEURISTIC**, not real TAP / a structure-based deamidation
predictor — use it to triage obvious liabilities **before** synthesis; confirm with the real tools for
any claim.

In [ ]:
from maturation_tools import developability_scan

# Scan the parent vs each top candidate; flag candidates that ADD a liability the parent didn't have.
parent_scan = developability_scan(parent)
parent_liab = set(parent_scan["liabilities"])
print("parent CDR liabilities (heuristic):", parent_scan["n_liabilities"], parent_scan["liabilities"])

rows = []
for v in vs:
    scan = developability_scan(v.sequence)
    introduced = set(scan["liabilities"]) - parent_liab
    rows.append(dict(design_id=v.design_id, mutation="+".join(v.mutations),
                     total_liabilities=scan["n_liabilities"],
                     introduced=";".join(f"{m}@{p}" for m, p in sorted(introduced)) or "(none)"))
liab_df = pd.DataFrame(rows)
print("\ncandidates that INTRODUCE a new CDR liability (avoid these):")
liab_df[liab_df["introduced"] != "(none)"].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(liab_df["total_liabilities"], bins=range(0, max(liab_df["total_liabilities"].max(), 1) + 2))
ax.set_xlabel("CDR liability motif count (heuristic)"); ax.set_ylabel("candidates")
ax.set_title("Developability liabilities per candidate (NOT real TAP)")
plt.tight_layout(); plt.savefig("results/developability.png", dpi=150); plt.show()
print("Prefer candidates that ADD no liability. Confirm with real TAP / deamidation tools before any claim.")

## 3 · Epistasis / combination reasoning `[extension]`

Single mutations are scored independently, but the bench tests *combinations*. **Epistasis** means the
effect of two mutations together is not the sum of their separate effects — they can reinforce or
interfere, especially when close in 3-D. You cannot get a real ΔΔG from the mock backend, so the
deliverable is the **reasoning + the plan**: take the few best single mutations, propose pairwise
combinations, re-score the pose (a real combo can break the interface even if both singles maintain it),
and flag pairs to test as a small block. Combinations to actually order stay FEW — the validation
budget is small.

In [ ]:
import itertools
from maturation_tools import assemble_candidate_set

# Take the best few single mutations (distinct positions) and propose pairwise combinations.
best_singles = assemble_candidate_set([v for v in vs], top_n=4)
combos = []
for a, b in itertools.combinations(best_singles, 2):
    ma, mb = a.mutations[0], b.mutations[0]
    # apply both to the parent (skip if they hit the same position)
    pa = int(ma[1:-1]); pb = int(mb[1:-1])
    if pa == pb:
        continue
    seq = parent
    try:
        seq = apply_mutation(apply_mutation(parent, ma), mb)
    except Exception:
        continue
    combos.append(Variant(design_id=f"EXAMPLE_DATA_COMBO_{ma}_{mb}", sequence=seq,
                          mutations=(ma, mb), source="combination"))
score_variants(combos, tool="mock")
combo_df = pd.DataFrame([dict(mutations="+".join(c.mutations), pae_interaction=c.pae_interaction,
                             scrmsd=c.scrmsd, n_liabilities=c.n_liabilities) for c in combos])
print("pairwise combinations to TEST (SYNTHETIC pose metrics; epistasis confirmed only by SPR):")
combo_df

## D3 (part 2) checklist
- [ ] Pose-maintenance figure (pae_interaction vs scRMSD vs parent) + the good-quadrant call.
- [ ] Developability liability scan: candidates that **introduce** a CDR liability flagged + dropped.
- [ ] Epistasis/combination reasoning: a FEW pairwise combos proposed + pose-rechecked; plan to test.
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as ranking/pose/plumbing, not affinity.

**Next:** `05_validation_plan.ipynb` — the SPR/DSF validation plan with controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — SPR kinetics, DSF stability, controls

**Standard slot:** *validation plan.* **For Project 15 this means:** because computational maturation
only *ranks* candidates, the deliverable is a **wet-lab plan that measures affinity for real** — express
the SMALL ranked variant set, measure **SPR/BLI kinetics** (kon/koff → KD, vs the parent's measured KD),
check **stability by DSF** (a higher-affinity variant that destabilizes the antibody is not a win), and
run it against mandatory **controls** (WT parent + a destabilizing decoy + a specificity panel) (D4/D5).

This generates structured plan files and a costed-reagent stub; it runs with no GPU. The deliverable
**D★** = a ranked affinity-improving CDR-mutation set + a developability scan + an SPR/DSF validation
plan with controls.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why a wet-lab assay (not "we predicted a tighter binder")

A computational maturation candidate is a **hypothesis**. ESM-1v/AbLang rank sequences; AF2-Multimer
checks the pose; **none of them measures affinity.** Most predicted affinity-improving mutations do
**not** validate. So the realistic pipeline is: **rank a small set in silico → express the variants →
measure KD by SPR/BLI against the parent's measured KD → confirm stability by DSF → keep only variants
that bind tighter without losing stability or specificity.** Frame your designs as **candidates to
test**, ranked, with the experiment attached.

## 1 · The SPR/BLI kinetics plan `[core]`

Surface plasmon resonance (or BLI) measures kon and koff → **KD**, directly comparable to the parent's
**measured literature KD**. The plan records the format, the analyte/ligand setup, the concentration
series, and the controls so it is reproducible and gradable.

In [ ]:
import json, os

spr_plan = {
    "assay": "SPR (e.g., Biacore) or BLI (e.g., Octet) kinetics: measure kon, koff -> KD",
    "candidate_set": "results/proj15_ranked.csv survivors (SMALL, pose-maintained, developable)",
    "baseline": "the PARENT antibody at its MEASURED literature KD (record the value + citation)",
    "setup": "capture antibody (or Fab) on the chip/biosensor; flow the antigen as analyte in a "
             "concentration series spanning ~0.1x-10x the expected KD",
    "readout": "global 1:1 kinetic fit -> kon, koff, KD per variant; compare KD_variant vs KD_parent",
    "success_criterion": "KD improved (lower) vs the parent by a pre-registered margin (state it), with "
                         "an acceptable fit and no avidity artifacts (use monovalent Fab if needed)",
    "controls": {
        "positive_baseline_WT": "the WT parent antibody — anchors the KD scale; every variant compared to it",
        "negative_destabilizing_decoy": "a deliberately destabilizing variant — EXPECTED to lose affinity/"
                                         "stability; proves the assay detects a loss, not just noise",
    },
    "expectation": "MOST predicted improvers will NOT validate. Report the hit rate (N improved / N tested), "
                   "not just the best clone.",
}
os.makedirs("results", exist_ok=True)
with open("results/spr_dsf_plan.json", "w") as fh:
    json.dump(spr_plan, fh, indent=2)
print("wrote results/spr_dsf_plan.json")
for k in ("assay", "baseline", "success_criterion", "expectation"):
    print(f"  {k}: {spr_plan[k]}")

## 2 · DSF stability + specificity counter-screen `[core]`

A tighter binder that **destabilizes** the antibody (lower melting temperature, more aggregation) is not
a developable win — so pair every SPR measurement with **DSF** (differential scanning fluorimetry → Tm)
and the developability liability scan from notebook 04. And confirm the mutation did not broaden
binding: run a **specificity panel** (the antigen vs a small set of off-target / related proteins) so an
"improved" variant is improved *for the right target*.

In [ ]:
dsf_specificity = {
    "stability_DSF": {
        "assay": "DSF (differential scanning fluorimetry) -> melting temperature Tm",
        "criterion": "variant Tm not meaningfully below the parent Tm (state the margin); flag any "
                     "aggregation; combine with the developability liability scan (notebook 04)",
    },
    "specificity_panel": {
        "purpose": "confirm the matured variant still prefers the intended antigen (no new off-target binding)",
        "panel": ["intended antigen (target)",
                  "a closely related off-target (e.g., a paralog/family member)",
                  "an irrelevant protein (e.g., BSA) as a non-specific-binding control"],
        "criterion": "variant binds the target but NOT the off-targets above the parent's background",
    },
    "decision": "keep variants that (a) improve KD vs parent, (b) hold Tm, (c) add no developability "
                "liability, and (d) stay target-specific — the rest are de-prioritised, honestly reported.",
}
import json
with open("results/dsf_and_specificity.json", "w") as fh:
    json.dump(dsf_specificity, fh, indent=2)
print(json.dumps(dsf_specificity, indent=2))

## 3 · Controls (mandatory) — WT + destabilizing decoy + specificity panel `[core]`

Controls are non-negotiable, even in the plan. `make_controls()` builds the two that anchor the affinity
assay: the **WT parent** (the measured-KD baseline every variant is compared to) and a **destabilizing
decoy** (a deliberately bad mutation expected to LOSE affinity/stability — it proves the assay can detect
a loss, so a measured improvement is real). The **specificity panel** (above) is the third control.

In [ ]:
from maturation_tools import example_parent_sequence, make_controls

parent = example_parent_sequence()
controls = make_controls(parent, antigen="ANTIGEN")
controls_record = {
    "positive_baseline_WT": {
        "design_id": controls[0].design_id,
        "role": "the un-mutated parent at its MEASURED KD — the reference for every variant",
    },
    "negative_destabilizing_decoy": {
        "design_id": controls[1].design_id,
        "mutation": "+".join(controls[1].mutations),
        "role": "EXPECTED to lose affinity/stability — proves the assay detects a loss; never reported "
                "as a maturation candidate",
    },
    "specificity_panel": "target vs a related off-target vs an irrelevant protein (see dsf_and_specificity.json)",
}
import json
with open("results/controls.json", "w") as fh:
    json.dump(controls_record, fh, indent=2)
print(json.dumps(controls_record, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
# EXAMPLE_DATA placeholders — replace with real vendor quotes + your institution's timeline.
plan_items = pd.DataFrame([
    dict(item="Gene synthesis of the variant set", purpose="express ranked candidates + controls", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Transient expression + purification", purpose="produce Fab/IgG variants", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant antigen (biotinylated)", purpose="SPR/BLI ligand/analyte", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI instrument time", purpose="kinetics: kon/koff -> KD", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="DSF reagents + plate reader time", purpose="Tm stability", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Specificity-panel proteins", purpose="off-target counter-screen", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic-antibody lead-optimization** project — maturing an existing antibody against its
(non-pathogen) target to raise affinity and keep developability. Dual-use risk is **low**: it improves a
therapeutic candidate, it does not create a novel hazard. In scope: therapeutic/diagnostic antibody
optimization. Out of scope: anything enhancing pathogen transmissibility/virulence, toxins, or designs
intended to cause harm. Any real gene-synthesis order must go through a biosecurity-screening provider
(IGSC member); wet-lab work requires institutional biosafety/ethics approval. Do not overstate
computational candidates as validated higher-affinity binders. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/spr_dsf_plan.json`: SPR/BLI kinetics plan vs the parent's MEASURED KD + success margin.
- [ ] `results/dsf_and_specificity.json`: DSF stability criterion + specificity counter-screen panel.
- [ ] `results/controls.json`: WT baseline + destabilizing decoy + specificity panel (all mandatory).
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Hit-rate framing: report N improved / N tested; most predicted improvers will not validate.
- [ ] Responsible-research framing stated.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — this project follows the antibody-family pattern (Project 17): light ESM-1v/AbLang +
ProteinMPNN scoring → AF2-Multimer pose check → developability → `design_type="antibody"` filter →
a SMALL ranked set + an SPR/DSF validation plan with controls.